# 🎨 Antigravity 4K WebGL Live Studio & GPU Batch Renderer (All-In-One UI Studio in Colab)
Studio Pembuat Video 4K Motion Graphics Lengkap dengan **Tampilan UI Visual Interaktif Langsung di Google Colab**!

### ✨ Fitur All-in-One:
1. **Tampilan UI Visual Langsung di Colab**: Atur warna, kecepatan, 24+ shader Paper & 3D ShaderGradient dengan slider & preview real-time.
2. **Tombol 1-Click Sync ke GPU**: Cukup buat antrean di UI lalu tekan **Simpan Antrean ke GPU**.
3. **GPU Multi-Worker Parallel Render**: Render 3-4 video 4K sekaligus secara paralel dan otomatis ter-download ke PC/Laptop!

## ⚙️ Step 1: Install Environment & GPU Dependencies (Jalankan Sekali)

In [ ]:
import os, subprocess, shutil
os.chdir('/content')

print("⏳ [1/3] Memasang Google Chrome & Library GPU Hardware...")
!wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i /tmp/chrome.deb > /dev/null 2>&1 || apt-get install -fy > /dev/null 2>&1
!apt-get install -y ffmpeg libgbm-dev libnss3 libasound2 zip > /dev/null 2>&1
!rm -f /tmp/chrome.deb

print("⏳ [2/3] Mengunduh Studio UI & Renderer terbaru dari GitHub...")
if os.path.exists('/content/shadergradientpaper'):
    shutil.rmtree('/content/shadergradientpaper', ignore_errors=True)

!git clone https://github.com/consistmaker/shadergradientpaper.git /content/shadergradientpaper

print("⏳ [3/3] Membangun Aplikasi WebGL UI & Engine...")
%cd /content/shadergradientpaper
!npm install --legacy-peer-deps > /dev/null 2>&1
!npm install puppeteer-core > /dev/null 2>&1
!npm run build
%cd /content

print("\n✅ ALL-IN-ONE STUDIO BERHASIL DISIAPKAN! SILAKAN LANJUT KE STEP 2.")

## 🌐 Step 2: Buka Antarmuka UI Visual Langsung di Colab
Jalankan cell di bawah ini. Anda bisa mengatur shader, menggeser slider warna, melihat preview real-time, dan mengklik **Export Batch** langsung di dalam panel Colab!

In [ ]:
from IPython.display import display, HTML

# Tampilkan UI Studio resmi yang responsif dan terintegrasi penuh langsung di notebook Colab
studio_url = "https://shadergradientpaper.vercel.app"

display(HTML(f'''
    <div style="border: 2px solid #6366f1; border-radius: 12px; overflow: hidden; margin-top: 10px; box-shadow: 0 10px 25px -5px rgba(99, 102, 241, 0.3);">
        <div style="background: #0f172a; color: #fff; padding: 12px 18px; font-weight: bold; font-size: 14px; display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid rgba(255,255,255,0.1);">
            <span style="display: flex; align-items: center; gap: 8px;">🎨 <b>Antigravity Live UI Studio (Full Interactive Visual Window)</b></span>
            <a href="{studio_url}" target="_blank" style="color: #38bdf8; text-decoration: none; font-size: 12px; background: rgba(56, 189, 248, 0.15); padding: 5px 10px; border-radius: 6px; border: 1px solid rgba(56,189,248,0.3);">↗ Buka Layar Penuh di Tab Baru</a>
        </div>
        <iframe src="{studio_url}" width="100%" height="800px" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture"></iframe>
    </div>
'''))

print("💡 PETUNJUK: Di dalam jendela UI di atas, pilih preset -> atur warna -> klik 'Export Batch' -> klik 'Copy JSON'. Lalu paste di Step 3 di bawah!")

## 📥 Step 3: Masukkan Resep JSON dari UI di Atas

In [ ]:
import json, os

# Paste JSON dari tombol 'Export Batch' UI Studio di sini:
RECIPE_JSON = '''
{
  "metadata": {
    "targetResolution": "3840x2160 (4K UHD)",
    "targetFps": 30,
    "loopDurationSeconds": 10,
    "isSeamlessLoop": true,
    "batchMode": "manual_queue"
  },
  "manualQueueList": [
    {
      "index": 1,
      "id": "item_1",
      "name": "Paper: mesh-gradient (#ffffff)",
      "engine": "paper",
      "config": {
        "shaderType": "mesh-gradient",
        "color1": "#ffffff",
        "color2": "#000000",
        "color3": "#ffffff",
        "color4": "#000000",
        "speed": 1.0,
        "distortion": 1.0,
        "swirl": 0.2
      }
    }
  ],
  "totalVideosInQueue": 1
}
'''

with open('/content/render_recipe.json', 'w') as f:
    f.write(RECIPE_JSON.strip())

recipe = json.loads(RECIPE_JSON)
print(f"🎯 Antrean Siap: {len(recipe.get('manualQueueList', []))} Video 4K UHD @ {recipe['metadata'].get('targetFps', 30)} FPS")
print("✅ Resep berhasil disimpan! Siap dieksekusi secara Paralel di Step 4.")

## 🚀 Step 4: Eksekusi GPU Multi-Worker Parallel Render & Auto-Download ke Laptop

In [ ]:
import subprocess, time, os, glob
from google.colab import files

output_dir = '/content/output_4k_videos'
os.makedirs(output_dir, exist_ok=True)
downloaded_files = set()

# Set 3-4 video paralel bersamaan untuk memaksimalkan 15GB VRAM GPU Nvidia T4
os.environ['CONCURRENCY'] = '3'

print("🚀 Memulai GPU Multi-Worker Parallel Render Engine...")
process = subprocess.Popen(
    ['node', '/content/shadergradientpaper/parallel_renderer.cjs'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1
)

while True:
    line = process.stdout.readline()
    if line:
        print(line, end='')
        if '✅ Success 4K Render:' in line:
            time.sleep(0.5)
            current_videos = glob.glob(f"{output_dir}/*.mp4")
            for v_path in current_videos:
                if v_path not in downloaded_files and os.path.exists(v_path):
                    file_size = (os.path.getsize(v_path) / (1024 * 1024))
                    print(f"   📥 [INSTANT AUTO-DOWNLOAD] Mengunduh: {os.path.basename(v_path)} ({file_size:.2f} MB)...")
                    try:
                        files.download(v_path)
                        downloaded_files.add(v_path)
                    except Exception as e:
                        pass

    if process.poll() is not None:
        for remaining in process.stdout.readlines():
            print(remaining, end='')
        break

print(f"\n🎉 SELESAI SEMPURNA! Seluruh video 4K telah dirender secara paralel dan otomatis ter-download ke laptop Anda.")